In [ ]:
#ABC Sampling simulation of Customer Churn 
#Do necessary imports#
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# 1. Setup and Observed Churn Data Generation
np.random.seed(42)  # For reproducibility

# True retention parameters (hidden from the ABC algorithm itself)
true_theta = 14.5   # True average customer lifespan in months
true_sigma = 3.0    # Fixed standard deviation
n_observations = 100

In [ ]:
# Generate simulated "observed" churn lifespans
observed_data = np.random.normal(true_theta, true_sigma, n_observations)
# Clip data at 0 since customer lifespans cannot logically be negative
observed_data = np.clip(observed_data, 0, None)

# Calculate observed summary statistics
obs_mean = np.mean(observed_data)
obs_std = np.std(observed_data)
obs_skew = stats.skew(observed_data)

In [ ]:
# 2. ABC Configuration
n_simulations = 50000  # Total samples drawn from prior
epsilon = 0.15         # Tolerance threshold for metric acceptance

# Prior boundaries for theta (1 month to 2-year contract lifespan)
prior_min, prior_max = 1.0, 24.0

# Storage for accepted samples
accepted_theta_scen1 = []
accepted_theta_scen2 = []
accepted_theta_scen3 = []

# 3. ABC Rejection Simulation Loop
for _ in range(n_simulations):
    # Sample a proposal theta from the uniform prior
    proposal_theta = np.random.uniform(prior_min, prior_max)
    
    # Generate simulated churn dataset using the proposed average lifespan
    sim_data = np.random.normal(proposal_theta, true_sigma, n_observations)
    sim_data = np.clip(sim_data, 0, None)
    
    # Compute summary statistics for the simulated dataset
    sim_mean = np.mean(sim_data)
    sim_std = np.std(sim_data)
    sim_skew = stats.skew(sim_data)
    
    # Scenario 1: Only the mean lifespan
    dist_1 = np.abs(sim_mean - obs_mean)
    if dist_1 < epsilon:
        accepted_theta_scen1.append(proposal_theta)
        
    # Scenario 2: Mean and Standard Deviation of lifetimes
    dist_2 = np.sqrt((sim_mean - obs_mean)**2 + (sim_std - obs_std)**2)
    if dist_2 < epsilon:
        accepted_theta_scen2.append(proposal_theta)
        
    # Scenario 3: Mean, Standard Deviation, and Skewness of lifetimes
    dist_3 = np.sqrt((sim_mean - obs_mean)**2 + (sim_std - obs_std)**2 + (sim_skew - obs_skew)**2)
    if dist_3 < epsilon:
        accepted_theta_scen3.append(proposal_theta)

In [ ]:
# 4. Plotting Posterior Densities
plt.figure(figsize=(12, 6))

plt.hist(accepted_theta_scen1, bins=30, alpha=0.5, label='Scenario 1: Mean Only', density=True)
plt.hist(accepted_theta_scen2, bins=30, alpha=0.5, label='Scenario 2: Mean + Std Dev', density=True)
plt.hist(accepted_theta_scen3, bins=30, alpha=0.5, label='Scenario 3: Mean + Std Dev + Skew', density=True)

plt.axvline(true_theta, color='black', linestyle='--', linewidth=2, label=f'True $\\theta$ ({true_theta} mo)')
plt.title('ABC Posterior Estimations for Churn Parameter $\\theta$')
plt.xlabel('$\\theta$ (Average Customer Lifespan in Months)')
plt.ylabel('Density')
plt.xlim(prior_min, prior_max)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Performance metrics
print(f"Scenario 1 accepted: {len(accepted_theta_scen1)} samples (Rate: {len(accepted_theta_scen1)/n_simulations:.4f})")
print(f"Scenario 2 accepted: {len(accepted_theta_scen2)} samples (Rate: {len(accepted_theta_scen2)/n_simulations:.4f})")
print(f"Scenario 3 accepted: {len(accepted_theta_scen3)} samples (Rate: {len(accepted_theta_scen3)/n_simulations:.4f})")